# Hallucination Detector: quick start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atomgptlab/hallucination_detector/blob/main/notebooks/quickstart.ipynb)

Check whether citations are real. Each reference is parsed into title,
authors, venue and year, matched against
[Semantic Scholar](https://www.semanticscholar.org/), and scored:

| Composite score | Status |
|---|---|
| ≥ 80 | **verified** |
| 60 – 80 | **partial** |
| < 60 | **likely hallucinated** |

The composite weights title 60%, authors 25% and venue 15%. Fabricated
references are caught mostly by the **author** term: an invented citation
often has a plausible title, but its authors never wrote a matching paper.

This notebook takes about two minutes:

1. Setup
2. Check structured references (no LLM needed)
3. Check a raw reference list or a PDF (uses an LLM to parse it)
4. Run a small benchmark slice
5. Explore the released benchmark results (offline)

Code and data: [github.com/atomgptlab/hallucination_detector](https://github.com/atomgptlab/hallucination_detector).

## 1. Setup

In [ ]:
import os, sys

if "google.colab" in sys.modules:
    if not os.path.exists("hallucination_detector"):
        !git clone -q https://github.com/atomgptlab/hallucination_detector.git
    %cd hallucination_detector
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    %cd ..          # running locally from a clone

print(os.getcwd())

### API keys (both optional)

* **Semantic Scholar key**: strongly recommended; without it requests are
  heavily rate-limited and some checks may come back empty.
  [Request one free](https://www.semanticscholar.org/product/api#api-key-form).
* **AtomGPT API key**: used by the LLM that parses raw citation strings
  (sections 3 and 4). Sign in at [atomgpt.org](https://atomgpt.org), then
  Profile → Settings → Account → API Keys. Without it, raw strings are parsed
  by a title-only regular expression, so authors and venue are not checked.

In Colab, add them under **Secrets** (key icon on the left) as
`SS_API_KEY` and `ATOMGPT_API_KEY`; otherwise you are prompted below
(press Enter to skip).

In [ ]:
from getpass import getpass


def get_secret(name, prompt):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return os.environ.get(name) or getpass(prompt + " (Enter to skip): ")


os.environ["SS_API_KEY"] = get_secret("SS_API_KEY", "Semantic Scholar API key") or ""
atomgpt_key = get_secret("ATOMGPT_API_KEY", "AtomGPT API key") or ""
if atomgpt_key:
    os.environ["LLM_URL"] = "https://atomgpt.org/api"
    # model used in the paper; any other open-weight model served by the
    # endpoint can be used instead (future versions may switch models)
    os.environ["LLM_MODEL"] = "openai/gpt-oss-20b"
    os.environ["LLM_API_KEY"] = atomgpt_key

# Configuration is read when the module is imported, so set keys first.
import logging
import pandas as pd
import hallucination as h

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
logging.getLogger("httpx").setLevel(logging.WARNING)
print("Semantic Scholar key:", "set" if h.SS_API_KEY else "not set")
print("LLM extraction:      ", h.LLM_MODEL or "not configured (regex fallback)")

## 2. Check structured references

`verify_reference` takes the fields directly, so no LLM is needed. These
are the two examples pre-filled in the
[web app](https://atomgpt.org/hallucination_detector): the first is the
real JARVIS paper; the second is a fabricated citation from the CiteAudit
benchmark (its title resembles a real paper, but those authors never wrote it).

In [ ]:
refs = [
    {
        "title": "The joint automated repository for various integrated "
                 "simulations (JARVIS) for data-driven materials design",
        "authors": ["Kamal Choudhary", "Kevin F Garrity", "Andrew CE Reid",
                    "Brian DeCost"],
        "journal": "npj Computational Materials",
        "year": "2020",
    },
    {
        "title": "Defense against adversarial attacks using spectral "
                 "regularization",
        "authors": ["Dinghuai Zhang", "Yang Song", "Inderjit Dhillon",
                    "Eric Xing"],
        "journal": "International Conference on Learning Representations",
        "year": "2020",
    },
]

rows = []
for ref in refs:
    r = h.verify_reference(ref)
    rows.append({
        "cited title": ref["title"],
        "status": r["status"],
        "score": r["score"],
        "title": r["title_score"],
        "authors": r["author_score"],
        "venue": r["journal_score"],
        "closest real paper": r["best_match"],
    })

pd.set_option("display.max_colwidth", 70)
pd.DataFrame(rows)

The fabricated entry still finds a real paper with a similar title, but
its author score is zero, which pulls the composite below the *verified*
threshold. Try editing the fields above and re-run.

## 3. Check a raw reference list or a PDF

`check_text` finds the references in free text (a pasted bibliography or
the text of a manuscript), parses them, and verifies each one. Parsing
uses the LLM if an AtomGPT key was given, otherwise the title-only regex.
Paste your own references below.

In [ ]:
text = """
References
1. Kamal Choudhary, Kevin F Garrity, Andrew CE Reid, Brian DeCost, et al. The
joint automated repository for various integrated simulations (JARVIS) for
data-driven materials design, npj computational materials 6 (1), 173 2020.
2. Dinghuai Zhang, Yang Song, Inderjit Dhillon, and Eric Xing. Defense against
adversarial attacks using spectral regularization. In ICLR, 2020.
3. Kamal Choudhary and Brian DeCost. Atomistic Line Graph Neural Network for
improved materials property predictions. npj Computational Materials, 7:185, 2021.
"""

results = h.check_text(text, sleep_sec=1.0)
pd.DataFrame(results)[
    ["index", "status", "score", "title_score", "author_score",
     "journal_score", "best_match"]
]

To check a whole manuscript, upload a PDF. The references section is
found automatically.

In [ ]:
RUN_PDF = False   # set True to upload a PDF

if RUN_PDF:
    from google.colab import files
    name = next(iter(files.upload()))
    pdf_results = h.check_text(h.extract_pdf_text(name), sleep_sec=1.0)
    df = pd.DataFrame(pdf_results)
    display(df["status"].value_counts())
    display(df[["index", "status", "score", "author_score", "reference"]])

## 4. Run a small benchmark slice

`run_benchmark.py` runs a labelled dataset and reports precision, recall
and the false-positive rate (positive class = fabricated; *flagged* =
anything other than *verified*). Here: 6 citations from the CiteAudit
sample bundled in `data/`. The full command line is in the README.

The runner stops on an authentication or quota error instead of scoring
the citation as a hallucination; re-running resumes where it stopped.

In [ ]:
import json
import run_benchmark

code = run_benchmark.main([
    "--file", "data/citeaudit_300.json",
    "--output", "demo_run.json",
    "--start", "0", "--limit", "6",
    "--workers", "1", "--no-resume",
])
if code == 2:
    print("Stopped on an API error (often the Semantic Scholar rate limit "
          "without a key). Re-run this cell to resume.")

run = json.load(open("demo_run.json"))
cols = ["index", "GT", "status", "score", "author_score", "citation"]
display(pd.DataFrame(run["results"])[[c for c in cols if c in run["results"][0]]]
        if run["results"] else "no results")
{k: run["summary"].get(k) for k in ("TP", "FP", "TN", "FN", "precision", "recall")}

## 5. Explore the released benchmark results (offline)

`results/` holds the per-citation outputs reported in the paper, so these
cells need no network:

* `neurips2025_v1.json`: 100 confirmed hallucinated citations from accepted
  NeurIPS 2025 papers (GPTZero); 94 flagged.
* `citeaudit_300_v2.json`: balanced CiteAudit sample, 150 fake and 149 real.

In [ ]:
import matplotlib.pyplot as plt

def load(path):
    d = json.load(open(path))
    return pd.DataFrame(d["results"]), d["summary"]

ca, ca_sum = load("results/citeaudit_300_v2.json")
ni, ni_sum = load("results/neurips2025_v1.json")

print("NeurIPS 2025 (all fake):", ni_sum["status"])
print("CiteAudit: precision %.2f, recall %.2f, false-positive rate %.2f" % (
    ca_sum["precision"], ca_sum["recall"], ca_sum["false_positive_rate"]))

order = ["verified", "partial", "likely hallucinated"]
ca["truth"] = ca["GT"].map({True: "real", False: "fake"})
pd.crosstab(ca["status"], ca["truth"]).reindex(order)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)
for ax, field in zip(axes, ["title_score", "author_score", "journal_score"]):
    for truth, color in [("real", "tab:blue"), ("fake", "tab:red")]:
        s = ca.loc[ca["truth"] == truth, field].dropna()
        ax.hist(s, bins=20, range=(0, 100), alpha=0.6, color=color,
                label=f"{truth} (n={len(s)})")
    ax.set_title(field.replace("_score", "").title() + " similarity")
    ax.set_xlabel("score")
axes[0].set_ylabel("citations")
axes[0].legend()
fig.suptitle("CiteAudit: per-field scores of real vs fake citations")
plt.tight_layout()
plt.show()

The author score separates real from fake far better than the title
score: fabricated titles often closely resemble real ones.

### Re-score under different weights

The scoring stage is deterministic, so other weights can be tested on the
stored per-field scores without new queries
(`scripts/rescore_sensitivity.py` does this for the paper's table).

In [ ]:
def rescore(df, w=(0.60, 0.25, 0.15), threshold=80.0):
    fields = ["title_score", "author_score", "journal_score"]
    def comp(row):
        pairs = [(wi, row[f]) for wi, f in zip(w, fields) if pd.notna(row[f])]
        return sum(wi * s for wi, s in pairs) / sum(wi for wi, _ in pairs)
    flagged = df.apply(comp, axis=1) < threshold
    fake = ~df["GT"].astype(bool)
    tp, fp = (flagged & fake).sum(), (flagged & ~fake).sum()
    fn = (~flagged & fake).sum()
    return {"weights": w, "precision": round(tp / (tp + fp), 2),
            "recall": round(tp / (tp + fn), 2),
            "FPR": round(fp / (~fake).sum(), 2)}

pd.DataFrame([rescore(ca, w) for w in
              [(0.60, 0.25, 0.15), (0.40, 0.45, 0.15), (0.30, 0.60, 0.10)]])

---

* Full documentation, benchmark runner flags and scoring definitions:
  [README](https://github.com/atomgptlab/hallucination_detector#readme)
* Web version: [atomgpt.org/hallucination_detector](https://atomgpt.org/hallucination_detector)